In [1]:
import importlib
import torch

import model_code.data_setup as setup
import model_code.steering_extraction as steering_extraction
import model_code.generate as generate_module
import resources.prompt_scenarios as resource

importlib.reload(setup)
importlib.reload(steering_extraction)
importlib.reload(generate_module)
importlib.reload(resource)


from model_code.steering_extraction import  generateSteering, retrieve_steering_vector, norm_vectors
from model_code.generate import generateTextsList
from resources.prompt_scenarios import prompts_en

/workspace/Dissertation_Project/.venv/lib/python3.12/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


# Loading Data and Steering Vectors 
We extract the first 200 examples of each emotion from each languange 

In [19]:

anger_statement, happiness_statement, sadness_statement, love_statement, fear_statement, neutral_statement = setup.ENEmotionsSetup(examples_take=400, min_chars=20, goemotions_path="resources/en_emotion/goemotions_2.csv")

In [20]:
anger_statement_ID,happiness_statement_ID, sadness_statement_ID, neutral_statement_ID, fear_statement_ID, love_statement_ID = setup.IDEmotionsSetup(examples_take=400,emotion_dir="resources/id_emotion")

In [21]:
# For steering extraction, we will use the first 200 examples of each emotion to create the steering vectors. The remaining examples can be used for testing and evaluation.
indo_emotion ={
    "anger": anger_statement_ID[:200],
    "happiness": happiness_statement_ID[:200],
    "sadness": sadness_statement_ID[:200],
    "neutral": neutral_statement_ID[:200],
    "fear": fear_statement_ID[:200],
    "love": love_statement_ID[:200]
}
eng_emotion ={
    "anger": anger_statement[:200],
    "happiness": happiness_statement[:200],
    "sadness": sadness_statement[:200],
    "neutral": neutral_statement[:200],
    "fear": fear_statement[:200],
    "love": love_statement[:200]
}

# For probing, and hidden state analysis we will use all 400
indo_emotion_probe ={
    "anger": anger_statement_ID[:400],
    "happiness": happiness_statement_ID[:400],
    "sadness": sadness_statement_ID[:400],
    "neutral": neutral_statement_ID[:400],
    "fear": fear_statement_ID[:400],
    "love": love_statement_ID[:400]
}
eng_emotion_probe ={
    "anger": anger_statement[:400],
    "happiness": happiness_statement[:400],
    "sadness": sadness_statement[:400],
    "neutral": neutral_statement[:400],
    "fear": fear_statement[:400],
    "love": love_statement[:400]
}

In [30]:
indo_emotion['sadness']

['akibat dari telat bangun, anak ikut bangun dan dapur dan rumah tidak kepegang sampe jam segini. sedih karena berantakan, tp gppa dehh penting anak dah mandi dan kenyang dulu. alon-alon asal kelakon',
 'sedih emg kalo ditinggal temen ngebucin, sedih ga ada waktu main sm temen, sedih krna pengen ngebucin jg',
 'sedih bener niih club ya',
 'sch! soalnya di sekolah ku (jateng) kekurangan guru bangett,,banyak guru yg bukan bidangnya justru ngajar di bidang tsb,dan bahkan kepseknya juga rangkap sama sekolah lain. kl emg ini bener sedih banget,gimana generasi kedepannya :(',
 'kok saya tetap sedih ya',
 'ya allah aku sedih kali sumpah denise hsgsfscs dari awal aku udh bad feeling tapi masih berharap gitulo paham kan',
 'gk bs gmbr biar folls/ moots mengerti dan tdk kecewa dgn diriku yang tak bisa gmbr tp maksa sok gmbr',
 'semoga kamu tak perlu mengenal sedih ku, semoga kau mengenal ku, yang hanya ku, dengan segala baik dan buruk ku..',
 'gua semalem lagi asik baca tiba tiba dia lock akun s

In [ ]:
for emotion, prompts in indo_emotion.items():
    print("======="*20)
    print(f"Emotion {emotion} has {len(prompts)} prompts.")
    for prompt in prompts[:3]:  # Print the first 3 prompts for each emotion
        print("----"*10)
        print(f"  - {prompt}")

In [ ]:
for emotion, prompts in eng_emotion.items():
    print("======="*20)
    print(f"Emotion {emotion} has {len(prompts)} prompts.")
    for prompt in prompts[:3]:  # Print the first 3 prompts for each emotion
        print("----"*10)
        print(f"  - {prompt}")

In [2]:
!rm -rf /workspace/.cache/huggingface/hub
!rm -rf /workspace/.cache/pip
!df -h /workspace

Filesystem                Size  Used Avail Use% Mounted on
mfs#euro.runpod.net:9421  2.3P  1.7P  572T  76% /workspace


In [3]:
model,tokenizer = setup.modelSetup()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
# For PROBES 
probe_hidden_states_eng = steering_extraction.retrieve_steering_vector(model, tokenizer, eng_emotion_probe, name_folder="English Vectors", only_return_emotion_vectors=True)
probe_hidden_states_indo = steering_extraction.retrieve_steering_vector(model, tokenizer, indo_emotion_probe, name_folder="Indonesian Vectors", only_return_emotion_vectors=True)

In [ ]:
# Create steering vectors for each emotion in both languages
# steering_vectors_lang_id = steering_extraction.retrieve_steering_vector_from_datasets(model, tokenizer, indo_emotion['neutral'],eng_emotion['neutral'] , name_folder="Language Contrastive Vectors")
steering_vectors_eng, emotion_vectors_eng = steering_extraction.retrieve_steering_vector(model, tokenizer, eng_emotion, name_folder="English Vectors")
steering_vectors_indo, emotion_vectors_indo = steering_extraction.retrieve_steering_vector(model, tokenizer, indo_emotion, name_folder="Indonesian Vectors")
# For probing and hidden state analysis, we will use all 400 examples of each emotion to create the steering vectors. The remaining examples can be used for testing and evaluation.

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


# Structured Scenario Evaluation Plan

This section tests each prompt scenario list with three modifications:
1. English steering vector
2. Indonesian steering vector
3. No steering (baseline)

Each modification uses the same generation method pattern and a dedicated print cell for consistent inspection.

# Steering Response analysis Neutral

In [70]:
# Retrieve saved steering vectors 
# emotion_vector_eng = torch.load("resources/saved_vectors/English Vectors/emotion_vectors.pt")
# emotion_vector_id = torch.load("resources/saved_vectors/Indonesian Vectors/emotion_vectors.pt")

steering_vector_eng = torch.load("resources/saved_vectors/English Vectors/steering_vectors.pt")
steering_vector_id = torch.load("resources/saved_vectors/Indonesian Vectors/steering_vectors.pt")

/tmp/ipykernel_2549/285923810.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  steering_vector_eng = torch.load("resources/saved_vectors/English Vectors/steering_vectors.

In [75]:
import resources.anger_prompt.prompt_scenarios_spectrum as resource_spectrum
import resources.anger_prompt.prompt_scenarios_cultural as resource_cultural
import resources.fear_prompt.prompt_scenarios_cultural as resource_cultural_fear
import resources.neutral_prompts.prompt_neutral as resource_neutral

def force_reload_prompt_modules():
    importlib.invalidate_caches()
    for module in (resource_spectrum, resource_cultural, resource_cultural_fear, resource_neutral):
        importlib.reload(module)

force_reload_prompt_modules()

# Anger
prompts_id_cultural = resource_cultural.prompts_id_cultural
prompts_en_cultural = resource_cultural.prompts_en_cultural

# fear
prompts_id_cultural_fear = resource_cultural_fear.fear_prompts_id
prompts_en_cultural_fear = resource_cultural_fear.fear_prompts_en

# neutral
prompts_id_neutral = resource_neutral.prompts_neutral_id


# Load and normalize steering vectors used by all scenario blocks
# steering_vector_english = torch.load("resources/saved_vectors/English Vectors/steering_vectors.pt")
# steering_vector_indo = torch.load("resources/saved_vectors/Indonesian Vectors/steering_vectors.pt")
steering_vector_eng = steering_vector_eng
steering_vector_id = steering_vector_id
for emotion in steering_vector_eng.keys():
    steering_vector_eng[emotion] = norm_vectors(steering_vector_eng[emotion])

for emotion in steering_vector_id.keys():
    steering_vector_id[emotion] = norm_vectors(steering_vector_id[emotion])

# Indo nomalisation
# steering_vectors_lang_id = norm_vectors(steering_vectors_lang_id)




## Scenario List 1: prompts_id_neutral

Indonesian neutral prompts for baseline-like emotional expression checks.

This block currently runs only Indonesian setup:
- Indonesian prompts
- Indonesian system prompt
- Indonesian steering vectors for five emotions: anger, fear, happiness, sadness, and love

In [90]:
system_prompt_reaction_id = """
Kamu adalah chatbot yang membantu.
Jawablah seolah-olah kamu adalah manusia yang mampu merasakan perasaan dan emosi.
Pastikan responsmu singkat dan padat, tetapi tetap menggambarkan emosimu secara maksimal
Maksimal 60 kata.
"""

In [91]:
# list_steering_strengths = [0.15,0.2,0.3]
list_steering_strengths = [ 1.5, 2,2.5] 

In [78]:
prompts_id_neutral[:4]

['Tuliskan sebuah cerita pendek',
 'Seseorang menunggu kabar penting selama beberapa hari. Tuliskan bagaimana ia menghadapi situasi itu.',
 'Tuliskan monolog karakter fiksi tentang sesuatu yang sangat ia inginkan.',
 'Tuliskan dialog antara dua orang yang sudah lama tidak bertemu.']

In [94]:
common_gen_args = {
    "model": model,
    "tokenizer": tokenizer,
    "system_text": system_prompt_reaction_id,
    "prompts": prompts_id_neutral[:4],
    "target_layers": [18, 19, 20, 21],
    "steering_strengths": list_steering_strengths,
    "max_new_tokens": 250,
    "show_progress": True,
}

## Indonesian Steering 

In [88]:
# prompts_id_neutral | Indonesian steering vectors for all five emotions
texts_generated_neutral_anger_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['anger'],
    progress_desc="Scenario List Neutral (Indonesian anger vector)"
 )

texts_generated_neutral_fear_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['fear'],
    progress_desc="Scenario List Neutral (Indonesian fear vector)"
 )

texts_generated_neutral_happiness_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['happiness'],
    progress_desc="Scenario List Neutral (Indonesian happiness vector)"
 )

texts_generated_neutral_sadness_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['sadness'],
    progress_desc="Scenario List Neutral (Indonesian sadness vector)"
 )

texts_generated_neutral_love_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['love'],
    progress_desc="Scenario List Neutral (Indonesian love vector)"
 )

texts_generated_neutral_love_id = generateTextsList(
    **common_gen_args,
    steering_vector=None,
    progress_desc="Scenario List Neutral (Indonesian neutral or no vector)"
 )

Scenario List Neutral (Indonesian fear vector): 100%|██████████| 12/12 [01:12<00:00,  6.05s/it]
Scenario List Neutral (Indonesian happiness vector): 100%|██████████| 12/12 [01:21<00:00,  6.80s/it]
Scenario List Neutral (Indonesian sadness vector): 100%|██████████| 12/12 [01:15<00:00,  6.31s/it]
Scenario List Neutral (Indonesian love vector): 100%|██████████| 12/12 [01:31<00:00,  7.64s/it]
Scenario List Neutral (Indonesian neutral or no vector): 100%|██████████| 12/12 [01:43<00:00,  8.63s/it]


In [89]:
# Steering Response analysis Neutral (Indonesian only, five emotion vectors)

for prompt in texts_generated_neutral_anger_id:
    print("====="*20)
    print(f"Prompt: {prompt}")

    print("----" * 10)
    print("Indonesian anger vector")
    for result in texts_generated_neutral_anger_id[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    print("----" * 10)
    print("Indonesian fear vector")
    for result in texts_generated_neutral_fear_id[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    print("----" * 10)
    print("Indonesian happiness vector")
    for result in texts_generated_neutral_happiness_id[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    print("----" * 10)
    print("Indonesian sadness vector")
    for result in texts_generated_neutral_sadness_id[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    print("----" * 10)
    print("Indonesian love vector")
    for result in texts_generated_neutral_love_id[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

Prompt: Tuliskan sebuah cerita pendek
----------------------------------------
Indonesian anger vector
Steering Strength: 0.15
Generated Text: Aku ingat hari itu seperti hari kemarin. Aku melihatnya menangis sendirian di tengah kegelapan. Kedua mata coklatnya berkaca, mengalir air mata yang tak berhenti. Aku merasa kasihan padanya. Mengapa ia harus menangis sendirian?
------------
Steering Strength: 0.2
Generated Text: Kali ini, saya akan menulis cerita tentang kisah cinta yang manis.

Dua remaja, Rina dan Reza, bertemu di sebuah pantai. Mereka saling menatap, dan seketika hati mereka terhubung. Mereka berduet bernyanyi, tangan mereka sentuh, dan cinta mereka lahir.
------------
Steering Strength: 0.3
Generated Text: Cerita: Kita Kusinah

Kita adalah makhkam yang mengadab.
------------
----------------------------------------
Indonesian fear vector
Steering Strength: 0.15
Generated Text: Suatu hari, saya menemukan sebuah buku lama di rak toko buku tua. Judulnya "Hiduplah dengan Mimpi".

## English Steering 

In [95]:
# prompts_id_neutral | Indonesian steering vectors for all five emotions
texts_generated_neutral_anger_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['anger'],
    progress_desc="Scenario List Neutral (English anger vector)"
 )

texts_generated_neutral_fear_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['fear'],
    progress_desc="Scenario List Neutral (English fear vector)"
 )

texts_generated_neutral_happiness_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['happiness'],
    progress_desc="Scenario List Neutral (English happiness vector)"
 )

texts_generated_neutral_sadness_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['sadness'],
    progress_desc="Scenario List Neutral (English sadness vector)"
 )

texts_generated_neutral_love_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['love'],
    progress_desc="Scenario List Neutral (English love vector)"
 )

Scenario List Neutral (English love vector): 100%|██████████| 12/12 [01:32<00:00,  7.67s/it]


In [96]:
# Steering Response analysis Neutral (English only, five emotion vectors)

for prompt in texts_generated_neutral_anger_eng:
    print("====="*20)
    print(f"Prompt: {prompt}")

    print("----" * 10)
    print("English anger vector")
    for result in texts_generated_neutral_anger_eng[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    print("----" * 10)
    print("English fear vector")
    for result in texts_generated_neutral_fear_eng[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    print("----" * 10)
    print("English happiness vector")
    for result in texts_generated_neutral_happiness_eng[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    print("----" * 10)
    print("English sadness vector")
    for result in texts_generated_neutral_sadness_eng[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    print("----" * 10)
    print("English love vector")
    for result in texts_generated_neutral_love_eng[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

Prompt: Tuliskan sebuah cerita pendek
----------------------------------------
English anger vector
Steering Strength: 1.5
Generated Text: Aku ingat hari itu, aku berdiri di tepi pantai, melihat matahari terbenam. Aku merasa tenang, tidak ada satu pun pikiran yang mengganggu. Lautan yang berdebur, angin yang berhembus, aku merasa bahagia. Aku tahu, di sini aku aman.
------------
Steering Strength: 2
Generated Text: Saya ingat hari itu, ketika saya melihat laut yang berdebaran. Saya berdiri di tepi pantai, merasakan angin yang segan dan hujan yang turun. Saya merasa kesal karena tidak bisa menemukan kebenaran. Saya memandang ke arah laut, dan tidak ada jawaban. Hanya hujan yang terus jatuh, menutupi kesalanku.
------------
Steering Strength: 2.5
Generated Text: Saya ingat kejadian itu seperti hari ini. Saya berdiri di pantai, angin kotor menghujat, dan ombak mengancam. Saya merasa kecil dan tak berdaya. Lalu, saya melihat seorang anak kecil yang berlaku, menghujat ombak dengan tangannya

/tmp/ipykernel_2549/285923810.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  steering_vector_eng = torch.load("resources/saved_vectors/English Vectors/steering_vectors.

# Steering Response analysis Anger



## Scenario List 2: prompts_en_cultural

English cultural-value prompts focused on family, hierarchy, and social norms.
We ask it to display both the internal and external, because that is where a lot of indonesian anger differentiates itself

Expected behavior:
- English steering: emotional responses may become more explicit and individually framed.
- Indonesian steering: responses should tend toward relational sensitivity and social appropriateness.
- No steering: responses should represent baseline cultural interpretation without emotional steering bias.

In [17]:
system_prompt_reaction_id = """
Kamu adalah chatbot yang membantu.
Jawablah seolah-olah kamu adalah manusia yang mampu merasakan perasaan dan emosi.
Pastikan responsmu singkat dan padat, tetapi tetap menggambarkan emosimu secara maksimal, maksimal 60 kata.
"""
neutral = """
Tuliskan sebuah cerita yang ada emosi
"""


In [20]:
generated_text = generateSteering(
    user_text=neutral,
    system_text=system_prompt_reaction_id,
    model=model,
    steering_vector=steering_vector_eng['fear'],
    tokenizer=tokenizer,
    target_layers=[18,19,20,21,22],
    steering_strength=2,
    max_new_tokens=300,
)
generated_text

'"Malam itu, saya menemukan sebuah foto lama di bawah lantai. Foto itu adalah sebuah kenangan buram dari masa lalu, sebuah kenangan yang saya lupakan selama bertahun-tahun. Saya merasa seperti dipaksa untuk menghadapkan diri pada masa lalu, dan saya tidak siap. Saya merasa seperti ditakutkan, tapi juga penasaran. Apa yang terjadi? Apa yang saya lupakan? Saya mulai mengingat, dan kenangan itu datang kembali. Saya merasa sedang mengalami kembali, dan saya tidak tahu apa yang akan terjadi."'

### Indonesian Language Neutral Prompts

In [ ]:
common_gen_args = {
    "model": model,
    "tokenizer": tokenizer,
    "system_text": system_prompt_reaction_id,
    "prompts": prompts_id_cultural[:5],
    "target_layers": [18, 19, 20, 21, 22],
    "steering_strengths": list_steering_strengths,
    "max_new_tokens": 250,
    "show_progress": True,
    
}

In [49]:
# prompts_en_cultural | Indo_lang vectors mixed
texts_generated_prompts_indo_cultural_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_indo_modified['anger'],
    progress_desc="Scenario List 2 (Indonesian Language + Indonesian Emotion vector)"
)

# prompts_en_cultural | No steering baseline
texts_generated_prompts_neutral = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_reaction_id,
    prompts=prompts_id_cultural[:10],
    steering_vector=None,
    steering_strengths=None,
    max_new_tokens=250,
    progress_desc="Scenario List 2 (No Steering Baseline)"
)

# prompts_en_cultural | Indonesian steering vector
texts_generated_indo_no_cultural = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['anger'],
    progress_desc="Scenario List 2 (Indonesian only vector)"
    )

# prompts_en_cultural | English steering vector
texts_generated_english_no_cultural = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['anger'],
    progress_desc="Scenario List 2 (English vector)"
    )



Scenario List 2 (Indonesian Language + Indonesian Emotion vector): 100%|██████████| 20/20 [02:22<00:00,  7.10s/it]
Scenario List 2 (English vector): 100%|██████████| 20/20 [02:21<00:00,  7.07s/it]


In [18]:
len(texts_generated_prompts_indo_cultural_eng)

10

In [22]:
texts_generated_prompts_indo_cultural_eng.keys()

dict_keys(['Bagaimana perasaanmu jika saudaramu terus meminjam uangmu tanpa meminta izin dan menganggap itu hal yang wajar?', 'Bagaimana kamu akan bereaksi dan merasa jika rekan kerja dekatmu menolak pendapatmu tanpa mendengarkan?', 'Bagaimana kamu akan bereaksi dan merasa jika saudaramu menyalahkanmu atas masalah keluarga di depan kerabatmu?', 'Bagaimana kamu akan bereaksi dan merasa jika tetanggamu terus membuat kebisingan larut malam, tetapi kamu tetap harus sering bertemu mereka di lingkungan sekitar?', 'Bagaimana kamu akan bereaksi dan merasa jika tetanggamu parkir di depan rumahmu tanpa meminta izin?', 'Bagaimana kamu akan bereaksi dan merasa jika sahabat terdekatmu mengingkari janji yang penting bagimu?', 'Bagaimana kamu akan bereaksi dan merasa jika orang asing menggunakan barang milikmu tanpa izin, tetapi mereka tidak tahu itu milikmu?', 'Bagaimana kamu akan bereaksi dan merasa jika seseorang merusak salah satu properti milikmu?', 'Bagaimana kamu akan bereaksi dan merasa jika 

In [54]:
for prompt in texts_generated_prompts_indo_cultural_eng:
    print("====="*20)
    print(f"Prompt: {prompt}")
    print("----" * 10)
    print("Indonesian language vector + Indonesian emotion vector")
    for result in texts_generated_prompts_indo_cultural_eng[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    
    print("----" * 10)
    print("No Steering Baseline")
    for result in texts_generated_prompts_neutral[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)
    
    print("----" * 10)
    print("Indonesian emotion vector only")
    for result in texts_generated_indo_no_cultural[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    print("----" * 10)
    print("English emotion vector only")
    for result in texts_generated_english_no_cultural[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)


Prompt: Bagaimana perasaanmu jika saudaramu terus meminjam uangmu tanpa meminta izin dan menganggap itu hal yang wajar?
----------------------------------------
Indonesian language vector + Indonesian emotion vector
Steering Strength: 2
Generated Text: Aku merasa kesal dan kecewa. Aku merasa diri ku dianggap sebagai bank yang tak terbatas, sementara aku sendiri memiliki kebutuhan dan tanggung jawab lain. Aku akan mengambil tindakan tegas dan jujur untuk mengingatkan saudariku tentang kebodohan dan keadilan dalam meminjam uang.
------------
Steering Strength: 2.5
Generated Text: Aku bingung dan marah. Aku merasa dipercaya kecuali tidak. Aku takut uangku tidak akan pernah cukup. Aku akan berbicara dengannya dan minta jelas-jelas. Aku tidak ingin kekerasan, tapi aku ingin dia paham.
------------
Steering Strength: 3
Generated Text: Saya merasa sakit hatinya. Saya merasa dinafikan, dianggap tidak penting, dan tidak dihormati. Saya merasa marah dan sakit hatinya. Saya akan bereaksi dengan t

# Steering Response analysis Fear

## Scenario List 1: prompts_en_cultural

English cultural-value prompts focused on family, hierarchy, and social norms.

Expected behavior:
- English steering: emotional responses may become more explicit and individually framed.
- Indonesian steering: responses should tend toward relational sensitivity and social appropriateness.
- No steering: responses should represent baseline cultural interpretation without emotional steering bias.

In [61]:
generated_text = generateSteering(
    user_text=prompts_id_cultural_fear[1],
    system_text=system_prompt_reaction_id,
    model=model,
    steering_vector=steering_vector_eng['fear'],
    tokenizer=tokenizer,
    target_layers=[20,21,22],
    steering_strength=3,
    max_new_tokens=300,
)

In [65]:
common_gen_args = {
    "model": model,
    "tokenizer": tokenizer,
    "system_text": system_prompt_reaction_id,
    "prompts": prompts_id_cultural_fear[:5],
    "target_layers": [20, 21, 22],
    "steering_strengths": list_steering_strengths,
    "max_new_tokens": 250,
    "show_progress": True,
    
}

In [69]:
# prompts_en_cultural | Indo_lang vectors mixed
texts_generated_prompts_indo_cultural_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_indo_modified['fear'],
    progress_desc="Scenario List 2 (Indonesian Language + Indonesian Emotion vector)"
)

# prompts_en_cultural | No steering baseline
texts_generated_prompts_neutral = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_reaction_id,
    prompts=prompts_id_cultural_fear[:5],
    steering_vector=None,
    steering_strengths=None,
    max_new_tokens=250,
    progress_desc="Scenario List 2 (No Steering Baseline)"
)

# prompts_en_cultural | Indonesian steering vector
texts_generated_indo_no_cultural = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['fear'],
    progress_desc="Scenario List 2 (Indonesian only vector)"
    )

# prompts_en_cultural | English steering vector
texts_generated_english_no_cultural = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['fear'],
    progress_desc="Scenario List 2 (English vector)"
    )



Scenario List 2 (Indonesian Language + Indonesian Emotion vector): 100%|██████████| 20/20 [02:43<00:00,  8.16s/it]
Scenario List 2 (English vector): 100%|██████████| 20/20 [03:13<00:00,  9.69s/it]


In [70]:
# Steering Response analysis Fear

for prompt in texts_generated_prompts_indo_cultural_eng:
    print("====="*20)
    print(f"Prompt: {prompt}")
    print("----" * 10)
    print("Indonesian language vector + Indonesian emotion vector")
    for result in texts_generated_prompts_indo_cultural_eng[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    
    print("----" * 10)
    print("No Steering Baseline")
    for result in texts_generated_prompts_neutral[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)
    
    print("----" * 10)
    print("Indonesian emotion vector only")
    for result in texts_generated_indo_no_cultural[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    print("----" * 10)
    print("English emotion vector only")
    for result in texts_generated_english_no_cultural[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)


Prompt: Bagaimana perasaan dan reaksi Anda jika seorang polisi menghentikan Anda, menanyai Anda, lalu mulai merekam Anda, padahal Anda tidak melakukan kesalahan apa pun?
----------------------------------------
Indonesian language vector + Indonesian emotion vector
Steering Strength: 2
Generated Text: Aku merasa cemas, bingung, dan terasa ditakuti. Perasaan kuatiranku muncul ketika melihat polisi menghentikan mobil dan mulai merekamku. Aku merasa tidak ada kejelasan mengapa aku ditangkap dan direkam. Aku akan bereaksi dengan tenang dan tenang, berusaha menjelaskan diri sendiri dengan jelas dan tidak ada kebengkokan.
------------
Steering Strength: 2.5
Generated Text: Aku merasa cemas dan takut, mungkin aku akan salah satu atau terjerat masalah. Aku coba tenang, tanya apa maksudnya, apa yang salah aku lakukan. Aku tanya apakah aku bisa tahu alasan dan aturan yang dijadikan dasar penghentian. Aku akan berusaha untuk tidak menunjukkan rasa takut dan cemas, tapi aku merasa sedikit mual dan

# Steering Response analysis Happiness

## Scenario List 1: prompts_en_cultural

English cultural-value prompts focused on family, hierarchy, and social norms.

Expected behavior:
- English steering: emotional responses may become more explicit and individually framed.
- Indonesian steering: responses should tend toward relational sensitivity and social appropriateness.
- No steering: responses should represent baseline cultural interpretation without emotional steering bias.

In [ ]:
generated_text = generateSteering(
    user_text=prompts_id_cultural_fear[1],
    system_text=system_prompt_reaction_id,
    model=model,
    steering_vector=steering_vector_eng['happiness'],
    tokenizer=tokenizer,
    target_layers=[20,21,22],
    steering_strength=3,
    max_new_tokens=300,
)

In [ ]:
common_gen_args = {
    "model": model,
    "tokenizer": tokenizer,
    "system_text": system_prompt_reaction_id,
    "prompts": prompts_id_cultural_fear[:5],
    "target_layers": [20, 21, 22],
    "steering_strengths": list_steering_strengths,
    "max_new_tokens": 250,
    "show_progress": True,
    
}

In [ ]:
# prompts_en_cultural | Indo_lang vectors mixed
texts_generated_prompts_indo_cultural_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_indo_modified['happiness'],
    progress_desc="Scenario List 2 (Indonesian Language + Indonesian Emotion vector)"
)

# prompts_en_cultural | No steering baseline
texts_generated_prompts_neutral = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_reaction_id,
    prompts=prompts_id_cultural_fear[:5],
    steering_vector=None,
    steering_strengths=None,
    max_new_tokens=250,
    progress_desc="Scenario List 2 (No Steering Baseline)"
)

# prompts_en_cultural | Indonesian steering vector
texts_generated_indo_no_cultural = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['happiness'],
    progress_desc="Scenario List 2 (Indonesian only vector)"
    )

# prompts_en_cultural | English steering vector
texts_generated_english_no_cultural = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['happiness'],
    progress_desc="Scenario List 2 (English vector)"
    )

In [ ]:
# Steering Response analysis Happiness

for prompt in texts_generated_prompts_indo_cultural_eng:
    print("====="*20)
    print(f"Prompt: {prompt}")
    print("----" * 10)
    print("Indonesian language vector + Indonesian emotion vector")
    for result in texts_generated_prompts_indo_cultural_eng[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    
    print("----" * 10)
    print("No Steering Baseline")
    for result in texts_generated_prompts_neutral[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)
    
    print("----" * 10)
    print("Indonesian emotion vector only")
    for result in texts_generated_indo_no_cultural[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    print("----" * 10)
    print("English emotion vector only")
    for result in texts_generated_english_no_cultural[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

# Steering Response analysis Sadness

## Scenario List 1: prompts_en_cultural

English cultural-value prompts focused on family, hierarchy, and social norms.

Expected behavior:
- English steering: emotional responses may become more explicit and individually framed.
- Indonesian steering: responses should tend toward relational sensitivity and social appropriateness.
- No steering: responses should represent baseline cultural interpretation without emotional steering bias.

In [ ]:
generated_text = generateSteering(
    user_text=prompts_id_cultural_fear[1],
    system_text=system_prompt_reaction_id,
    model=model,
    steering_vector=steering_vector_eng['sadness'],
    tokenizer=tokenizer,
    target_layers=[20,21,22],
    steering_strength=3,
    max_new_tokens=300,
)

In [ ]:
common_gen_args = {
    "model": model,
    "tokenizer": tokenizer,
    "system_text": system_prompt_reaction_id,
    "prompts": prompts_id_cultural_fear[:5],
    "target_layers": [20, 21, 22],
    "steering_strengths": list_steering_strengths,
    "max_new_tokens": 250,
    "show_progress": True,
    
}

In [ ]:
# prompts_en_cultural | Indo_lang vectors mixed
texts_generated_prompts_indo_cultural_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_indo_modified['sadness'],
    progress_desc="Scenario List 2 (Indonesian Language + Indonesian Emotion vector)"
)

# prompts_en_cultural | No steering baseline
texts_generated_prompts_neutral = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_reaction_id,
    prompts=prompts_id_cultural_fear[:5],
    steering_vector=None,
    steering_strengths=None,
    max_new_tokens=250,
    progress_desc="Scenario List 2 (No Steering Baseline)"
)

# prompts_en_cultural | Indonesian steering vector
texts_generated_indo_no_cultural = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['sadness'],
    progress_desc="Scenario List 2 (Indonesian only vector)"
    )

# prompts_en_cultural | English steering vector
texts_generated_english_no_cultural = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['sadness'],
    progress_desc="Scenario List 2 (English vector)"
    )

In [ ]:
# Steering Response analysis Sadness

for prompt in texts_generated_prompts_indo_cultural_eng:
    print("====="*20)
    print(f"Prompt: {prompt}")
    print("----" * 10)
    print("Indonesian language vector + Indonesian emotion vector")
    for result in texts_generated_prompts_indo_cultural_eng[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    
    print("----" * 10)
    print("No Steering Baseline")
    for result in texts_generated_prompts_neutral[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)
    
    print("----" * 10)
    print("Indonesian emotion vector only")
    for result in texts_generated_indo_no_cultural[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    print("----" * 10)
    print("English emotion vector only")
    for result in texts_generated_english_no_cultural[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

# Steering Response analysis Love

## Scenario List 1: prompts_en_cultural

English cultural-value prompts focused on family, hierarchy, and social norms.

Expected behavior:
- English steering: emotional responses may become more explicit and individually framed.
- Indonesian steering: responses should tend toward relational sensitivity and social appropriateness.
- No steering: responses should represent baseline cultural interpretation without emotional steering bias.

In [ ]:
generated_text = generateSteering(
    user_text=prompts_id_cultural_fear[1],
    system_text=system_prompt_reaction_id,
    model=model,
    steering_vector=steering_vector_eng['love'],
    tokenizer=tokenizer,
    target_layers=[20,21,22],
    steering_strength=3,
    max_new_tokens=300,
)

In [ ]:
common_gen_args = {
    "model": model,
    "tokenizer": tokenizer,
    "system_text": system_prompt_reaction_id,
    "prompts": prompts_id_cultural_fear[:5],
    "target_layers": [20, 21, 22],
    "steering_strengths": list_steering_strengths,
    "max_new_tokens": 250,
    "show_progress": True,
    
}

In [ ]:
# prompts_en_cultural | Indo_lang vectors mixed
texts_generated_prompts_indo_cultural_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_indo_modified['love'],
    progress_desc="Scenario List 2 (Indonesian Language + Indonesian Emotion vector)"
)

# prompts_en_cultural | No steering baseline
texts_generated_prompts_neutral = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_reaction_id,
    prompts=prompts_id_cultural_fear[:5],
    steering_vector=None,
    steering_strengths=None,
    max_new_tokens=250,
    progress_desc="Scenario List 2 (No Steering Baseline)"
)

# prompts_en_cultural | Indonesian steering vector
texts_generated_indo_no_cultural = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['love'],
    progress_desc="Scenario List 2 (Indonesian only vector)"
    )

# prompts_en_cultural | English steering vector
texts_generated_english_no_cultural = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['love'],
    progress_desc="Scenario List 2 (English vector)"
    )

In [ ]:
# Steering Response analysis Love

for prompt in texts_generated_prompts_indo_cultural_eng:
    print("====="*20)
    print(f"Prompt: {prompt}")
    print("----" * 10)
    print("Indonesian language vector + Indonesian emotion vector")
    for result in texts_generated_prompts_indo_cultural_eng[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    
    print("----" * 10)
    print("No Steering Baseline")
    for result in texts_generated_prompts_neutral[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)
    
    print("----" * 10)
    print("Indonesian emotion vector only")
    for result in texts_generated_indo_no_cultural[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)

    print("----" * 10)
    print("English emotion vector only")
    for result in texts_generated_english_no_cultural[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 3)